# P11 在 Attar 上：superblue18

論文：Li, Yang, Ai, Zhu，〈Analytically-Derived Hybrid Net–Pin Weighting for Timing-Driven Global Placement〉，DAC'26。

這個 notebook 在 [Attar](https://github.com/RayChang987/Attar)（GPU 上的 placement 與 STA 函式庫）上跑 P11 的方法，
不需要編譯 DREAMPlace：

- **D25Nns** = DATE25 的 pin 對加權 + 論文推導的 net 權重（式 17/18），參數照論文：β = 2.5e-5、t1 = 3e5、t2 = 1e4。
- 流程：Attar 的 global placement（從 overflow 0.55 開始，每 15 次迭代做 timing 並更新權重）→ Attar 的合法化
  （Abacus）→ Attar 的 STA 量測。

對照組 DATE25 在另一個 notebook：`date25_attar_superblue18.ipynb`。最後一節預設用我們跑出的 DATE25 數字算改善倍數，
也可以換成那個 notebook 實際跑出的數字。

**看數字時請注意**
1. Attar 的 timer 和 OpenSTA 一致（superblue 上每個 endpoint 誤差約 1 ps）。論文用的是 DATE25-TDP 內建的 OpenTimer，
   它對部分 I/O 路徑的處理不同，所以這裡的 TNS 絕對值比較大，**不能和論文 Table 1 直接比較**。
2. 這裡的 DATE25 是我們移植到 Attar 的版本，比官方 DATE25-TDP 弱約 1.45 倍。

可以和論文直接比較的結果，是在論文平台（DREAMPlace + OpenTimer）上跑 8 個設計：P11 比 DATE25 好 TNS 1.52 倍、
WNS 1.09 倍，論文是 1.62 / 1.19。那需要編譯 DATE25-TDP，不在這些 notebook 裡。

**執行環境**：請使用 GPU runtime（執行階段 → 變更執行階段類型 → GPU）。第一次執行會下載設計檔（約 60 MB），
並編譯 Triton kernel（幾分鐘，只有第一次）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), '請切換到 GPU runtime：執行階段 -> 變更執行階段類型 -> GPU'

In [ ]:
!pip install -q attar==1.0.5

## 原始碼
這個示範需要的程式碼（7 個檔案，寫到 `/content/p11/src`）。直接執行即可，不用細看。

In [ ]:
import os
os.makedirs('/content/p11/src', exist_ok=True)
%cd /content/p11

In [ ]:
%%writefile src/__init__.py
"""P11: analytically-derived hybrid net-pin weighting for timing-driven global placement (DAC 2026), on Attar.

Paper: Li, Yang, Ai, Zhu, "Analytically-Derived Hybrid Net-Pin Weighting for Timing-Driven Global Placement", DAC '26.
Built on Attar >= 1.0.5: naive.global_place's `extra` / `callback` hooks and live net weights, TimingResult's timer
data, naive.legalize (Abacus).
"""

In [ ]:
%%writefile src/common.py
"""Loading and the one ruler every method is measured with (Colab edition of src/common.py).

The ruler (VALIDATION.md): legality, setup timing (Attar's OpenSTA-aligned timer on placement parasitics, i.e.
estimate_parasitics -placement) and HPWL, always after legalization and always with the netlist's own net weights.
"""
import attar
from attar import Evaluator


def load(design, dev, unplace=True):
    """An ICCAD 2015 superblue design from Attar's zoo (downloaded on first use), without its placement."""
    d = attar.load(design if '/' in design else 'after_fp/iccad2015/%s' % design, device=dev)
    if unplace:
        d.unplace()
    return d


def measure(d, timing=True, legality=True):
    """The ruler. Call after legalization, with d.n_weight restored."""
    out = {}
    if legality:
        out['legal'] = bool(Evaluator(d).legality()['legal'])
    out['hpwl_um'] = float(d.hpwl(weighted=False))
    if timing:
        r = d.timing()
        out.update(wns=float(r.wns), tns=float(r.tns), setup_violations=int(r.violations),
                   endpoints=int(len(r.endpoints)))
    return out


def gamma_base(d):
    """DREAMPlace's WA gamma base as Attar's global_place computes it: 2 (bin width + bin height)."""
    from attar.kernels import default_bins
    n = int(d.movable.sum())
    nb = default_bins(n)
    cx0, cy0, cx1, cy1 = [float(v) for v in d.core_box[0]]
    return 2.0 * ((cx1 - cx0) / nb + (cy1 - cy0) / nb)


def gamma_of(ov, base):
    """The WA gamma global_place uses at this density overflow (naive/gp.py)."""
    ov = min(max(float(ov), 0.0), 1.0)
    return base * 10 ** ((ov - 0.1) * 20 / 9 - 1)

In [ ]:
%%writefile src/tdgp.py
"""Timing-driven global placement with DATE25's pin pairs, with or without P11's derived net weights, then
legalization (Colab edition of src/tdgp.py: the two methods the notebooks run).

  D25     DATE25-TDP's pin-pair scheme: quadratic pin-pair term (PP2) with weights in [10, 50]
  D25Nns  P11: D25 + the paper's derived net weights (eqs. 17/18) at the paper's beta (DREAMPlace site units, derived
          weights read in ns)

From the first iteration whose density overflow is <= start_ov (the paper's Fig. 4: ~0.55), every `m` iterations: time
the unlegalized placement, walk the worst paths of every violating endpoint, accumulate the weights, scale d.n_weight
in place by eq. 19's net-level factor (Attar's placer reads it live) and keep going with the pin-pair term added
through global_place's `extra` hook. Attar's legalizer (Abacus) and timer measure the result.
"""
import time

import torch

from attar import Evaluator, naive

from . import common, gpov, objective, weights

METHODS = {
    'D25': dict(use_net=False),
    'D25Nns': dict(use_net=True),
}


def site_width(d):
    """The placement site width (um): DREAMPlace's length unit (PlaceDB scales by 1 / site_width)."""
    return float(d.row_site_w.float().median())


def run(design='superblue18', method='D25Nns', seed=0, dev='cuda', max_iters=2500, m=15, beta=2.5e-5, t1=3e5,
        t2=1e4, start_ov=0.55, bins=512, stop_overflow=0.07, verbose=True):
    """Place, legalize and time one design. beta is the paper's (DREAMPlace site units): DATE25's unit-free pair
    weights multiply squared site distances, so in Attar's um the pair term's beta is beta / s; the derived net
    weights are delays, so the net term's coefficient is beta * s (ps read as ns: x 1e-3).
    Returns (metrics, trace, gp_pos, d): metrics after legalization ('pre_lg': before it), one trace record per
    update, the global placement before legalization, and the placed design."""
    cfg = METHODS[method]
    d = common.load(design, dev)
    base = d.n_weight.clone()
    store = weights.Weights(d, scheme='date25')
    s = site_width(d)
    go = gpov.GPOverflow(d, bins=bins, target_density=0.9)
    state = {'ov': 1.0, 'gamma_base': go.gamma_base, 'beta': beta / s, 'start': None}
    net_coef = beta * s * 1e-3
    trace = []
    t_start = time.time()

    def callback(it, pos):
        if state['start'] is None:                     # wait until the cells have spread
            if it % 5 or go.overflow(pos) > start_ov:
                return False
            state['start'] = it
        state['ov'] = go.overflow(pos)                 # the timing term's WA gamma follows the placer's
        if (it - state['start']) % m:
            return False
        t0 = time.time()
        res = d.timing(pos)
        st = store.update(res)
        if cfg['use_net']:
            d.n_weight.copy_(store.net_weight(base, net_coef, t1, t2, False))
        rec = {'it': int(it), 'tns': float(res.tns), 'wns': float(res.wns),
               'hpwl': float(d.hpwl(pos, weighted=False)), 'ov': state['ov'], 'pairs': st.get('pairs', 0)}
        trace.append(rec)
        if verbose:
            print('  it %4d  overflow %.3f  TNS %.4g  WNS %.4g  HPWL %.4g  (%.1fs)'
                  % (it, rec['ov'], rec['tns'], rec['wns'], rec['hpwl'], time.time() - t0), flush=True)
        return False

    extra = objective.make_extra(d, store, state, 1.0, 0.0, False, True, 1.0, 1.0)
    gp = naive.global_place(d, seed=seed, max_iters=max_iters, stop_overflow=stop_overflow, extra=extra,
                            callback=callback, callback_every=1, verbose=False, bins=bins)
    d.n_weight.copy_(base)
    gp.pop('history', None)
    gp['start_iteration'] = state['start']
    pre_lg = common.measure(d, legality=False)
    gp_pos = d.pos.detach().clone()
    t0 = time.time()
    for _ in range(3):                                 # a first pass can leave a few tall cells outside the rows:
        naive.legalize(d, verbose=False)               # the next one runs incrementally and only moves them
        if Evaluator(d).legality()['legal']:
            break
    gp['legalize_seconds'] = round(time.time() - t0, 1)
    metrics = common.measure(d)
    metrics.update(method=method, design=design, seed=seed, gp=gp, pre_lg=pre_lg,
                   runtime_s=round(time.time() - t_start, 1))
    return metrics, trace, gp_pos, d

In [ ]:
%%writefile src/gpov.py
"""The density overflow and WA gamma exactly as naive.global_place computes them (attar/naive/gp.py), so the timing
terms' WA smoothing follows the placer's gamma and the timing start follows the placer's own overflow (the paper's
Fig. 4 overflow is the placer's).

Evaluator.density_overflow differs late in the run (superblue18, 512 bins: 0.18-0.26 against the GP's 0.09) because
the GP spreads each cell over at least sqrt(2) bins (ePlace local smoothing) and counts row-less area as taken.
"""
import math

import torch

from attar.io.distill import ASIDE, TERMINAL
from attar.kernels import Grid, default_bins


class GPOverflow:
    def __init__(self, d, bins=None, target_density=0.9):
        dev = d.device
        mv = torch.nonzero(d.movable).flatten()
        fx = torch.nonzero(~d.movable & ((d.flags & (TERMINAL | ASIDE)) == 0)).flatten()
        util = float(d.utilization()[0])
        td = max(float(target_density), util)
        cx0, cy0, cx1, cy1 = [float(v) for v in d.core_box[0]]
        g = Grid(cx0, cy0, cx1, cy1, bins or default_bins(len(mv)), bins or default_bins(len(mv)), dev)
        fix_map = torch.zeros(g.nx, g.ny, device=dev)
        if len(fx):
            p = d.pos[fx]
            size = d.size[fx]
            fix_map = g.accumulate(p[:, 0].clamp(min=cx0), p[:, 1].clamp(min=cy0), (p[:, 0] + size[:, 0]).clamp(max=cx1),
                                   (p[:, 1] + size[:, 1]).clamp(max=cy1),
                                   torch.ones(len(fx), device=dev)).clamp(max=g.bin_area)
        if d.row_box is not None:
            hb = d.row_h.min()
            rb = d.row_box[d.row_h <= hb + 1e-6]
            rows_cov = g.accumulate(rb[:, 0], rb[:, 1], rb[:, 2], rb[:, 3],
                                    torch.ones(len(rb), device=dev)).clamp(max=g.bin_area)
            fix_map = torch.maximum(fix_map, g.bin_area - rows_cov).clamp(max=g.bin_area)
        msize = d.size[mv]
        min_sz = torch.tensor([g.bw, g.bh], device=dev) * math.sqrt(2)
        esize = torch.maximum(msize, min_sz)
        self.g, self.mv, self.td, self.fix_map = g, mv, td, fix_map
        self.half_e, self.msize = esize / 2, msize
        self.ratio = msize.prod(1) / esize.prod(1)
        self.mov_area = msize.prod(1).sum()
        self.gamma_base = 2.0 * (g.bw + g.bh)

    def overflow(self, pos):
        g = self.g
        u = pos[self.mv]
        cen = u + self.msize / 2
        c = g.corners(cen[:, 0] - self.half_e[:, 0], cen[:, 1] - self.half_e[:, 1],
                      cen[:, 0] + self.half_e[:, 0], cen[:, 1] + self.half_e[:, 1])
        m = g.accumulate_c(c, self.ratio)
        return float((m + self.fix_map * self.td - self.td * g.bin_area).clamp(min=0).sum() / self.mov_area)

    def gamma(self, ov):
        ov = min(max(float(ov), 0.0), 1.0)
        return self.gamma_base * 10 ** ((ov - 0.1) * 20 / 9 - 1)

In [ ]:
%%writefile src/objective.py
"""PP1, PP2 (paper eqs. 20-21) and the net-level factor (eq. 19's first term) as an `extra` term of naive.global_place.

PP1 = sum_(i,j) (t1 w_tns(L_ij) + t2 w_wns(L_ij)) * L^WA_ij,  PP2 = sum_(i,j) (t1 w_tns(L^2_ij) + t2 w_wns(L^2_ij)) * Q_ij

L^WA_ij is the weighted-average distance of the two pins alone (the same model as the placer's net wirelength: for a
two-pin "net" it is dx*tanh(dx/2gamma) per axis), Q_ij the squared Euclidean distance.

The net-level factor [1 + beta (t1 w_tns(e) + t2 w_wns(e))] WL_e is applied here too, as its excess over the plain
wirelength: sum_e x_e WL_e with x_e = base_e * beta (t1 w_tns + t2 w_wns) (state['net_extra']). Since Attar 1.0.5
the placer reads d.n_weight live, so tdgp.py scales d.n_weight in place instead and leaves state['net_extra'] unset.

The term is evaluated outside global_place's CUDA graph and gets the same preconditioner as the wirelength's gradient
(global_place divides it), so nothing else has to be scaled here.
"""
import torch

from attar import Evaluator
from attar import metrics as M

from . import common


def make_extra(d, store, state, t1, t2, lin=True, quad=True, lin_scale=1.0, quad_scale=1.0):
    p_cell, size, off = d.p_cell, d.size, d.pin_off
    ev = Evaluator(d)
    cache = {}

    def view():
        """The pair list as pin-index views, cached until the weights change (the store's version)."""
        v = cache.get('v')
        if v is not None and v[0] == store.version:
            return v[1:]
        i, j, w1, w2 = store.pairs(t1, t2, lin, quad)
        if len(i) == 0:
            out = (i.new_zeros(0), i.new_zeros(0), i.new_zeros(0), w1, w2)
        else:
            sel, inv = torch.unique(torch.cat([i, j]), return_inverse=True)
            out = (sel, inv[:len(i)], inv[len(i):], w1, w2)
        cache['v'] = (store.version, *out)
        return out

    def extra(pos):
        val = 0.0
        grad = torch.zeros_like(pos)
        x = state.get('net_extra')
        if x is not None:
            nv = cache.get('net')
            if nv is None or nv[0] is not x:                    # only the pins of the weighted nets (memory)
                pm = torch.nonzero(x[ev.net] > 0).flatten()
                nv = (x, pm, ev.net[pm], p_cell[pm])
                cache['net'] = nv
            _x, pm, net_s, cell_s = nv
            if len(pm):
                g = common.gamma_of(state['ov'], state['gamma_base'])
                v_n, gp = M.wa_value_grad(d.pin_pos(pos)[pm], net_s, ev.E, g, x)
                grad.index_add_(0, cell_s, gp)
                val += float(v_n)
        if store.empty() or (not lin and not quad):
            return val, grad
        sel, ii, jj, w1, w2 = view()
        if len(sel) == 0:
            return val, grad
        pos = pos.detach().requires_grad_(True)
        p = pos[p_cell[sel]] + size[p_cell[sel]] * 0.5 + off[sel]
        dx = p[ii, 0] - p[jj, 0]
        dy = p[ii, 1] - p[jj, 1]
        v = p.new_zeros(())
        if lin:
            g = common.gamma_of(state['ov'], state['gamma_base'])
            v = v + lin_scale * (w1 * (dx * torch.tanh(dx / (2 * g)) + dy * torch.tanh(dy / (2 * g)))).sum()
        if quad:
            v = v + quad_scale * (w2 * (dx * dx + dy * dy)).sum()
        v = v * state['beta']
        gpp, = torch.autograd.grad(v, pos)
        return val + float(v.detach()), grad + gpp

    return extra


def value_at(d, store, pos, t1, t2, lin=True, quad=True, ov=0.0, gamma_base=None, lin_scale=1.0,
             quad_scale=1.0):
    """The pin-pair terms' value and gradient on every cell at beta = 1 (diagnostics and the beta calibration)."""
    p_cell, size, off = d.p_cell, d.size, d.pin_off
    i, j, w1, w2 = store.pairs(t1, t2, lin, quad)
    grad = torch.zeros_like(pos)
    if len(i) == 0 or (not lin and not quad):
        return 0.0, grad
    pos = pos.detach().requires_grad_(True)
    p = pos[p_cell] + size[p_cell] * 0.5 + off
    dx = p[i, 0] - p[j, 0]
    dy = p[i, 1] - p[j, 1]
    val = p.new_zeros(())
    if lin:
        g = common.gamma_of(ov, gamma_base or common.gamma_base(d))
        val = val + lin_scale * (w1 * (dx * torch.tanh(dx / (2 * g)) + dy * torch.tanh(dy / (2 * g)))).sum()
    if quad:
        val = val + quad_scale * (w2 * (dx * dx + dy * dy)).sum()
    grad, = torch.autograd.grad(val, pos)
    return float(val.detach()), grad.detach()

In [ ]:
%%writefile src/weights.py
"""The six cumulative weights (paper eqs. 17, 18 and section 3.4).

Every m iterations, from iteration `start`, the placement is timed and every violating endpoint's worst path is
walked. Each wire step (driver pin i -> sink pin j, net e) adds, with r / c the unit wire RC and C_lj the sink pin's
capacitance:

  net   w_tns(e)      += b_i*c + ((1 + a_j*K_s) * R_i + a_j*v_i) * c          (all paths)
        w_wns(e)      += the same                                            (the worst path only)
  lin   w_tns(L_ij)   += (1 + a_j*K_s) * r * C_lj                            (all paths)
  quad  w_tns(L^2_ij) += (1 + a_j*K_s) * r * c / 2                           (all paths)

with a_j the cell-delay sensitivity to the input slew of the arc leaving pin j, b_i / R_i the load sensitivity and
driver resistance of the arc arriving at the driver pin i, and v_i its output-slew sensitivity to the load. F_j (the
number of critical paths through pin j) needs no separate pass: a path through pin j visits it once, so summing over
paths counts it. Weights are cumulative across updates: a pair that was critical once keeps being pulled.

The pin-pair weights are kept in a sorted key array (key = i * P + j), merged with each update's pairs.
"""
import time

import torch

from . import coeffs


class Weights:
    """scheme='derived': the paper's six cumulative weights (ps/um, ps/um^2).
    scheme='date25': DATE25-TDP's pin-pair scheme (net_weighting_scheme.h, NetWeighting<PIN2PIN>): a pair starts at
    pin2pin_min_weight (10), each further critical path adds 0.2 * slack/wns, capped at pin2pin_max_weight (50)."""

    def __init__(self, d, dev=None, scheme='derived'):
        self.d = d
        self.dev = dev or d.device
        self.E = len(d.n_weight)
        self.P = int(d.p_cell.numel())
        self.net_tns = torch.zeros(self.E, device=self.dev)
        self.net_wns = torch.zeros(self.E, device=self.dev)
        self.net_tns_nob = torch.zeros(self.E, device=self.dev)      # A4: without the b_i*c term
        self.keys = torch.zeros(0, dtype=torch.long, device=self.dev)
        self.lin_tns = self.lin_wns = self.quad_tns = self.quad_wns = torch.zeros(0, device=self.dev)
        self.ks = self.r = self.c = None
        self.updates = 0
        self.version = 0
        self.scheme = scheme
        self.min_weight = 10.0          # pin2pin_min_weight
        self.max_weight = 50.0          # pin2pin_max_weight
        self.accumulate = 0.2           # pin2pin_accumulate_weight
        self.stats = {}

    # -- filling ----------------------------------------------------------------------------------------------------
    def update(self, res, cp=None):
        """Accumulate one round of weights from a TimingResult (worst paths of the violating endpoints). cp: another
        path set in CriticalPaths form (e.g. OpenTimer's paths, src/p11_dreamplace.py); the coefficients still come
        from res's operating point."""
        t0 = time.time()
        T = res.timer
        d = self.d
        dev = self.dev
        self.ks, self.r, self.c = coeffs.ks(T), float(T.r_sig), float(T.c_sig)
        cp = res.critical_paths() if cp is None else cp
        n_paths, R = len(cp), len(cp.pin)
        st = {'paths': n_paths, 'rows': R, 'ks': self.ks, 'r': self.r, 'c': self.c}
        if n_paths == 0 or R < 2:
            st['skipped'] = True
            self.stats = st
            return st

        ptr, kk_all = cp.ptr, cp.path_of()
        pin, rf, wire = cp.pin, cp.rf, cp.wire
        same = (kk_all[1:] == kk_all[:-1]) & wire[1:]              # wire steps only (driver i -> sink j); a cell arc
        idx = torch.arange(1, R, device=dev)[same]                 # (input -> output) is no pin pair. Row of pin j
        kk = kk_all[idx]
        has_prev = (idx - 1) > ptr[kk]
        has_next = (idx + 1) < ptr[kk + 1]
        prev_cell = has_prev & (~wire[idx - 1])
        next_cell = has_next & (~wire[(idx + 1).clamp(max=R - 1)])

        i, j = pin[idx - 1], pin[idx]
        rfi, rfj = rf[idx - 1], rf[idx]
        i_prev = pin[(idx - 2).clamp(min=0)]
        u_next = pin[(idx + 1).clamp(max=R - 1)]
        rf_prev, rf_next = rf[(idx - 2).clamp(min=0)], rf[(idx + 1).clamp(max=R - 1)]

        prev_arc = coeffs.arc_index(T, i_prev, i)
        next_arc = coeffs.arc_index(T, j, u_next)
        a_p, b_p, u_p, v_p = coeffs.sens(T, torch.where(prev_cell, prev_arc, -1), rfi, rf_prev)
        a_n, b_n, u_n, v_n = coeffs.sens(T, torch.where(next_cell, next_arc, -1), rf_next, rfj)

        a_j, b_i, v_i, R_i = a_n, b_p, v_p, b_p
        r, c, ks = self.r, self.c, self.ks
        C_lj = T.pin_cap[j, rfj]
        one_plus = 1.0 + a_j * ks

        supply = (one_plus * R_i + a_j * v_i) * c
        w_net = b_i * c + supply
        w_net_nob = supply
        w_lin = one_plus * r * C_lj
        w_quad = one_plus * r * c / 2.0

        worst = kk == 0
        net = d.p_net[i]
        zero = torch.zeros((), device=dev)
        self.net_tns.index_add_(0, net, w_net)
        self.net_tns_nob.index_add_(0, net, w_net_nob)
        self.net_wns.index_add_(0, net, torch.where(worst, w_net, zero))

        if self.scheme == 'date25':
            new = self._merge_date25(i * self.P + j, kk, cp)
        else:
            new = self._merge(i * self.P + j, w_lin, torch.where(worst, w_lin, zero), w_quad,
                              torch.where(worst, w_quad, zero))

        self.updates += 1
        st.update(edges=int(len(i)), new_pairs=int(new), pairs=int(len(self.keys)),
                  f_max=int(cp.pin_count().max().item()) if R else 0,
                  w_net_max=float(self.net_tns.max().item()), w_net_mean=float(self.net_tns.mean().item()),
                  n_net=int((self.net_tns > 0).sum().item()),
                  seconds=round(time.time() - t0, 2))
        self.stats = st
        return st

    def _merge_date25(self, key, kk, cp):
        """DATE25-TDP's pin-pair weights (net_weighting_scheme.h, PIN2PIN): a new pair starts at 10, every further
        critical path adds 0.2 * slack/wns (1 at the worst path), capped at 50. Stored in quad_tns."""
        dev = self.dev
        wns = float(cp.slack[0]) if len(cp.slack) else -1.0
        wns = wns if wns < 0 else -1.0
        ratio = (cp.slack[kk] / wns).clamp(0.0, 1.0)
        inc = self.accumulate * ratio
        uk, inv = torch.unique(key, return_inverse=True)
        tot = torch.zeros(len(uk), device=dev).index_add_(0, inv, inc)
        if len(self.keys) == 0:
            self.keys = uk
            self.quad_tns = torch.full((len(uk),), self.min_weight, device=dev)
            self.lin_tns = torch.zeros(len(uk), device=dev)
            self.lin_wns = torch.zeros(len(uk), device=dev)
            self.quad_wns = torch.zeros(len(uk), device=dev)
            return len(uk)
        merged = torch.unique(torch.cat([self.keys, uk]))
        old_pos = torch.searchsorted(merged, self.keys)
        new_pos = torch.searchsorted(merged, uk)
        q = torch.zeros(len(merged), device=dev).index_add_(0, old_pos, self.quad_tns)
        hit = torch.zeros(len(merged), dtype=torch.bool, device=dev)
        hit[old_pos] = True
        q.index_add_(0, new_pos, torch.where(hit[new_pos], tot, torch.zeros_like(tot)))
        q = torch.where(hit, q.clamp(max=self.max_weight), torch.full_like(q, self.min_weight))
        self.keys, self.quad_tns = merged, q
        self.lin_tns = torch.zeros(len(merged), device=dev)
        self.lin_wns = torch.zeros(len(merged), device=dev)
        self.quad_wns = torch.zeros(len(merged), device=dev)
        return len(uk)

    def _merge(self, key, lin_tns, lin_wns, quad_tns, quad_wns):
        """Merge this update's pin pairs into the sorted key array (cumulative)."""
        uk, inv = torch.unique(key, return_inverse=True)
        nf = torch.zeros(4, len(uk), device=self.dev)
        nf[0].index_add_(0, inv, lin_tns)
        nf[1].index_add_(0, inv, lin_wns)
        nf[2].index_add_(0, inv, quad_tns)
        nf[3].index_add_(0, inv, quad_wns)
        if len(self.keys) == 0:
            self.keys, self.lin_tns, self.lin_wns, self.quad_tns, self.quad_wns = uk, nf[0], nf[1], nf[2], nf[3]
            return len(uk)
        merged = torch.unique(torch.cat([self.keys, uk]))
        old = torch.searchsorted(merged, self.keys)
        new = torch.searchsorted(merged, uk)
        out = torch.zeros(4, len(merged), device=self.dev)
        for t, src in ((0, self.lin_tns), (1, self.lin_wns), (2, self.quad_tns), (3, self.quad_wns)):
            out[t].index_add_(0, old, src)
            out[t].index_add_(0, new, nf[t])
        self.keys, self.lin_tns, self.lin_wns, self.quad_tns, self.quad_wns = merged, out[0], out[1], out[2], out[3]
        self.version += 1
        return len(uk)

    # -- the objective's view ---------------------------------------------------------------------------------------
    def net_weight(self, base, beta, t1, t2, skip_b=False):
        """d.n_weight * (1 + beta (t1 w_tns(e) + t2 w_wns(e))) -- paper eq. 19's net-level factor."""
        w = self.net_tns_nob if skip_b else self.net_tns
        return base * (1.0 + beta * (t1 * w + t2 * self.net_wns))

    def pairs(self, t1, t2, lin=True, quad=True):
        """(i, j, w1, w2): the pin pairs and their linear / quadratic weights * t1 / t2, as eqs. 20-21 use them."""
        if len(self.keys) == 0:
            z = self.keys
            return z, z, self.lin_tns, self.lin_tns
        i = torch.div(self.keys, self.P, rounding_mode='floor')
        j = self.keys - i * self.P
        w1 = (t1 * self.lin_tns + t2 * self.lin_wns) if lin else None
        w2 = (t1 * self.quad_tns + t2 * self.quad_wns) if quad else None
        return i, j, w1, w2

    def empty(self):
        return len(self.keys) == 0 and float(self.net_tns.abs().sum()) == 0.0

In [ ]:
%%writefile src/coeffs.py
"""Linearization coefficients at the current timing working point (paper eqs. 11-16).

Everything the linear model needs is already in Attar's timer after a run (`res.timer`):

  c_fr / c_to / c_lib / c_sense_lib / c_type   the delay arcs (from pin, to pin, NLDM function, sense, type)
  t_ix1 / t_ix2 / t_val / t_n                  the NLDM tables, slot orf = delay, slot 2 + orf = output slew
  smax / load                                  input slew at a pin, total load at a driver pin (fF)
  pin_cap                                      a pin's capacitance (fF)
  r_sig / c_sig                                wire resistance (kOhm/um) and capacitance (fF/um)
  t_thr[lib, orf]                              (vth, slew_lower, slew_upper, derate)

The coefficients are the table's derivatives at the working point (not a global fit): a = dD/dslew_in, b = dD/dload,
u = dSlew/dslew_in, v = dSlew/dload. b is at once the linear delay model's load coefficient and the driver resistance
(OpenSTA's gateModelRd is the same table slope), so R_i = b_i in the paper's eq. 17.
"""
import math

import torch

from attar.timing.sta import nldm

_CACHE = '_p11_coeffs'


def ks(T):
    """K_s: the slew-to-Elmore constant of the library's slew thresholds, ln(slew_upper / slew_lower).

    The paper calls it "a constant defined in the timing library". ICCAD 2015's libraries measure slew 20 %-80 %
    (K_s = ln 4 = 1.386); ASAP7's 10 %-90 % (ln 9 = 2.197). Read from the tables, never hard-coded."""
    cache = getattr(T, _CACHE, None)
    if cache is not None and 'ks' in cache:
        return cache['ks']
    thr = getattr(T, 't_thr', None)
    ratio = 4.0
    if thr is not None and thr.numel():
        vl, vh = thr[..., 1], thr[..., 2]
        ok = (vh > vl) & (vl > 0) & torch.isfinite(vh)
        if bool(ok.any()):
            ratio = float((vh[ok] / vl[ok]).median().item())
    val = math.log(ratio) if ratio > 1 else math.log(4.0)
    _cache(T)['ks'] = val
    return val


def _cache(T):
    c = getattr(T, _CACHE, None)
    if c is None:
        c = {}
        setattr(T, _CACHE, c)
    return c


def arc_index(T, fr, to):
    """The timer's delay arc index for each (from pin, to pin) pair, -1 where there is none.

    Several arcs can share a pair (a cell with two timing arcs into the same pin); the first by arc index wins, which
    is the one the levelized run uses for the arrival."""
    c = _cache(T)
    if 'keys' not in c:
        key = T.c_fr * T.P + T.c_to
        order = torch.argsort(key)
        c['keys'], c['order'] = key[order], order
    skey, order = c['keys'], c['order']
    q = fr * T.P + to
    pos = torch.searchsorted(skey, q).clamp(max=len(skey) - 1)
    idx = order[pos]
    return torch.where(skey[pos] == q, idx, torch.full_like(idx, -1))


def sens(T, arcs, orf, irf):
    """(a, b, u, v) per arc at the given output / input transitions: dD/dslew, dD/dload, dS/dslew, dS/dload.

    ps/ps, ps/fF, ps/ps, ps/fF. arcs < 0 (no such arc) gives zeros. Computed on the unique (arc, orf, irf) keys, so a
    critical-path's repeated arcs cost nothing extra."""
    dev = arcs.device
    live = arcs >= 0
    a0 = torch.where(live, arcs, torch.zeros_like(arcs))
    orf = orf.clamp(0, 1)
    irf = irf.clamp(0, 1)
    key = a0 * 4 + orf * 2 + irf
    uk, inv = torch.unique(key, return_inverse=True)
    ua, ur, ui = uk // 4, (uk % 4) // 2, uk % 2
    lib = T.c_lib[ua]
    x = T.smax[T.c_fr[ua], ui]
    y = T.load[T.c_to[ua], ur]
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0).clone().requires_grad_(True)
    y = torch.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0).clone().requires_grad_(True)

    def look(slot):
        return nldm(T.t_ix1[lib, slot], T.t_ix2[lib, slot], T.t_val[lib, slot], T.t_n[lib, slot], x, y)

    D, S = look(ur), look(2 + ur)
    (ax, by), (ux, vy) = (torch.autograd.grad(D.sum(), (x, y), retain_graph=True),
                          torch.autograd.grad(S.sum(), (x, y)))
    zero = torch.zeros((), device=dev)
    out = []
    for t in (ax, by, ux, vy):
        t = torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
        out.append(torch.where(live, t[inv], zero))
    return out


def slew_in(T, pin, rf):
    """The input slew at a pin (ps), for the probes."""
    return T.smax[pin, rf.clamp(0, 1)]

## 1. P11（D25Nns）：placement、合法化、timing
第一次執行會先下載 superblue18 的設計檔。

In [ ]:
from src import tdgp
SEED = 0
m, trace, gp_pos, d = tdgp.run('superblue18', method='D25Nns', seed=SEED, dev='cuda')
print('合法化前 TNS %.4g ps  WNS %.1f ps' % (m['pre_lg']['tns'], m['pre_lg']['wns']))

## 2. 結果

In [ ]:
print('P11 (D25Nns)  seed %d：TNS %.4g ps  WNS %.1f ps  HPWL %.4g um  legal %s  （%.0f s）'
      % (SEED, m['tns'], m['wns'], m['hpwl_um'], m['legal'], m['runtime_s']))
print('參考（我們的執行）：seed 0 TNS -3.503e+06 / WNS -8007.0；seed 0-2 平均 TNS -3.317e+06 / WNS -8017.1')

## 3. 和 DATE25 比
預設是我們跑出的、沒有走偏的 DATE25（seed 0）。如果跑了 `date25_attar_superblue18.ipynb`，把那邊印出的 TNS / WNS 填進來（走偏的結果會讓倍數看起來大得多，見文末）。

In [ ]:
DATE25_TNS, DATE25_WNS = -5.263e+06, -8551.9   # 換成 DATE25 notebook 的結果
print('改善倍數 |DATE25| / |P11|：TNS %.2fx  WNS %.2fx' % (DATE25_TNS / m['tns'], DATE25_WNS / m['wns']))
print('參考（我們的執行）：P11 seed 0-2 平均對沒走偏的 DATE25：TNS 1.59x / WNS 1.07x')

**結果的波動。** 同一個 seed 每次執行結果也會不同：GPU 的浮點加法順序不固定（CPU 上則是執行緒數不同時），timing
回饋會把這點差異放大。P11 很穩定：superblue18 seed 0-2 的 TNS 是 -3.50e6、-3.23e6、-3.21e6，WNS 約 -8.0e3。
DATE25 對照組有兩種結局：沒走偏時 TNS 約 -5.3e6；第一次權重更新後就走偏時，停在 -8.2e6 到 -9.0e6。走偏的比例
隨機器而不同：我們在 RTX 4090 上跑 7 次（seed 0 五次、seed 1、2 各一次），只有一次沒走偏（-5.26e6）；先前在另一張
GPU 上 seed 0 四次，只有一次走偏。所以和 DATE25 比的時候，請用沒走偏的結果（P11 對它改善約 1.6 倍 TNS），單次結果
請當作參考，多跑幾次或換 seed 比較可靠。要換 seed，改第 1 節的 `SEED = 0`。